# County Rainfall and Planting Advisory

**PLP Data Analysis Assignment 2**

## 1. Project overview

This notebook cleans monthly rainfall data for Nairobi, Nakuru, Kisumu, Mombasa and Machakos, analyses county-level rainfall patterns with NumPy, and produces a CSV summary to support an NGO's farmer SMS advisories.

**Analysis period:** January–December (12 monthly observations per county).  
**Rainfall unit:** millimetres (mm).

## 2. Import libraries and define the dataset

We use:
- `numpy` for numerical arrays, aggregation, Boolean masks and locating maximum values.
- `csv` for writing the final summary file.
- `calendar` for readable month names.

In [1]:
import numpy as np
import csv
import calendar

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = list(calendar.month_name)[1:]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Number of counties:", len(counties))
print("Number of months:", len(months))

Number of counties: 5
Number of months: 12


## 3. Clean the Machakos rainfall data

The Machakos observations were supplied as text. The cleaning function:
1. Removes leading and trailing spaces using `strip()`.
2. Converts valid entries to integers.
3. Catches `ValueError` specifically when an entry cannot be converted (for example, `"N/A"` or an empty string).
4. Replaces invalid entries with the rounded mean of all valid Machakos observations.

The imputation mean is calculated from valid observations only, before any missing values are replaced. This avoids using imputed values to calculate the value used for imputation.

In [2]:
def clean_machakos(raw_values, month_names):
    valid_values = []
    invalid_months = []

    for value, month in zip(raw_values, month_names):
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            invalid_months.append(month)

    imputed_value = round(np.mean(valid_values))

    cleaned_values = []
    for value in raw_values:
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(imputed_value)

    print(f"Valid-observation mean: {np.mean(valid_values):.2f} mm")
    print(f"Imputed value (rounded): {imputed_value} mm")
    print("Months imputed:", ", ".join(invalid_months))

    return cleaned_values

machakos_clean = clean_machakos(machakos_raw, months)
print("Cleaned Machakos rainfall data:", machakos_clean)

Valid-observation mean: 67.50 mm
Imputed value (rounded): 68 mm
Months imputed: April, September
Cleaned Machakos rainfall data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 4. Combine the county data into one NumPy array

Each row represents a county and each column represents a month. The resulting array should therefore have five rows and twelve columns (5 × 12).

In [3]:
rainfall_rows = clean_rows + [machakos_clean]
R = np.array(rainfall_rows)

print("Rainfall array:")
print(R)
print("Array shape:", R.shape)
print("Array dtype:", R.dtype)

Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array dtype: int64


## 5. Annual rainfall totals and monthly averages

- `axis=1` adds values across each row, giving one annual total per county.
- `axis=0` averages down each column, giving one average for each calendar month across the five counties.

In [4]:
annual_totals = np.sum(R, axis=1)
monthly_averages = np.mean(R, axis=0)

print("Annual rainfall totals (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total:.0f} mm")

print("\\nAverage rainfall by month across all five counties (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals (mm):
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm
\nAverage rainfall by month across all five counties (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## 6. Wettest month in each county

`np.argmax(R, axis=1)` returns the column index of the largest monthly rainfall value in each county's row. We use those indices to retrieve the month names. If a county has a tie, `argmax` returns the first occurrence.

In [5]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_indices]
wettest_rainfall = np.max(R, axis=1)

print("Wettest month by county:")
for county, month, amount in zip(counties, wettest_months, wettest_rainfall):
    print(f"{county}: {month} ({amount} mm)")

Wettest month by county:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


## 7. Count dry months

A dry month is defined in this assignment as a month with rainfall **below 50 mm**. The Boolean expression `R < 50` creates a True/False mask. Summing the mask along `axis=1` counts the dry months in each county because NumPy treats `True` as 1 and `False` as 0.

In [6]:
dry_month_mask = R < 50
dry_month_counts = np.sum(dry_month_mask, axis=1)

print("Number of dry months (rainfall below 50 mm):")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Number of dry months (rainfall below 50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## 8. Identify the recommended two-month planting window

For each county, we add each month to the month immediately after it. The expression `R[:, :-1] + R[:, 1:]` creates eleven adjacent two-month totals per county (January–February through November–December).

`np.argmax(..., axis=1)` identifies the starting month of the highest-rainfall pair in each county. This is a rainfall-based indicator for advisory purposes, not a guarantee of suitable planting conditions.

In [7]:
two_month_totals = R[:, :-1] + R[:, 1:]
planting_start_indices = np.argmax(two_month_totals, axis=1)
planting_windows = [
    f"{months[index]}-{months[index + 1]}"
    for index in planting_start_indices
]
highest_two_month_rainfall = np.max(two_month_totals, axis=1)

print("Recommended two-month rainfall windows:")
for county, window, total in zip(counties, planting_windows, highest_two_month_rainfall):
    print(f"{county}: {window} ({total} mm combined)")

Recommended two-month rainfall windows:
Nairobi: April-May (395 mm combined)
Nakuru: April-May (265 mm combined)
Kisumu: April-May (420 mm combined)
Mombasa: April-May (390 mm combined)
Machakos: November-December (278 mm combined)


## 9. Create the rainfall summary CSV

The CSV contains exactly one row per county and the five required columns. We use Python's built-in `csv` module and `csv.DictWriter` to write the file.

In [8]:
fieldnames = [
    "county",
    "annual_total_mm",
    "wettest_month",
    "dry_months",
    "planting_window"
]

with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for i, county in enumerate(counties):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(annual_totals[i]),
            "wettest_month": wettest_months[i],
            "dry_months": int(dry_month_counts[i]),
            "planting_window": planting_windows[i]
        })

print("Created rainfall_summary.csv")

Created rainfall_summary.csv


## 10. Planting Advisory

The NGO can use the highest-rainfall consecutive two-month period in each county as an initial rainfall-based guide when preparing location-specific planting messages. The analysis identifies **April–May for Nairobi, April–May for Nakuru, April–May for Kisumu, April–May for Mombasa, and November–December for Machakos** as the respective highest-rainfall two-month windows. Farmers should prepare land and obtain suitable seed ahead of the expected window, while timing actual planting to the onset of dependable rains and local agricultural guidance. The NGO should avoid presenting these windows as guaranteed planting dates because monthly totals do not show the timing, intensity or distribution of rainfall within each month. In addition, the Machakos April and September values were missing or invalid and were replaced with the rounded mean of the valid observations. The dataset covers only one year's monthly readings, so it may not represent long-term rainfall patterns or year-to-year climate variability.


## 11. Data Quality Notes

- Machakos had two invalid observations: April (N/A) and September (empty).
- Both were replaced with 68 mm, the rounded mean of the ten valid observations.
- Other rainfall readings were assumed valid but were not independently verified.
- Only one year of data was available, limiting conclusions about long-term rainfall patterns.
- Planting windows are based on monthly rainfall totals and do not account for soil conditions, crop requirements or rainfall distribution within each month.


## 12. Final checks

The checks below confirm that the array has the required dimensions, that the summary has one record per county, and that the CSV contains the required column names.

In [9]:
assert R.shape == (5, 12), "Expected a 5 by 12 rainfall array."
assert len(counties) == 5
assert len(annual_totals) == 5
assert len(planting_windows) == 5

with open("rainfall_summary.csv", "r", newline="", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    csv_rows = list(reader)
    assert reader.fieldnames == fieldnames
    assert len(csv_rows) == 5

print("All final checks passed.")
print("CSV data rows:", len(csv_rows))
print("CSV columns:", fieldnames)

All final checks passed.
CSV data rows: 5
CSV columns: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
